In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoder, TransformerDecoderLayer
import re
from tqdm import tqdm
import numpy as np
from matplotlib import pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

c:\Repositories\proai\course9-genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_ds = load_dataset("parquet", data_files="data/train.parquet")['train']
test_ds = load_dataset("parquet", data_files="data/test.parquet")['train']

In [3]:
MAX_SENT_LEN = 15
vocabulary_test = set()

inputs_test, outputs_test = [], []

for i in tqdm(range(len(test_ds))):

  inputs_sent, dialog_sent = ["<sos>"], ["<sos>"]

  inputs_sent += re.findall(r"\w+", ''.join(test_ds[i]['dialog'][0]))
  dialog_sent += re.findall(r"\w+", ''.join(test_ds[i]['dialog'][1:]))

  # change to lowercase
  inputs_sent = [x.lower() for x in inputs_sent]
  dialog_sent = [x.lower() for x in dialog_sent]

  inputs_sent.append("<eos>")
  dialog_sent.append("<eos>")

  if len(inputs_sent) >= MAX_SENT_LEN:
    inputs_sent = inputs_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(inputs_sent)):
      inputs_sent.append("<pad>")

  if len(dialog_sent) >= MAX_SENT_LEN:
    dialog_sent = dialog_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(dialog_sent)):
      dialog_sent.append("<pad>")

  # add parsed sentences
  inputs_test.append(inputs_sent)
  outputs_test.append(dialog_sent)

  vocabulary_test.update(inputs_sent)
  vocabulary_test.update(dialog_sent)

vocabulary = list(vocabulary_test)

100%|██████████| 1000/1000 [00:00<00:00, 4113.50it/s]


In [4]:
MAX_SENT_LEN = 15
vocabulary = set()

inputs, outputs = [], []

for i in tqdm(range(len(train_ds))):

  inputs_sent, dialog_sent = ["<sos>"], ["<sos>"]

  inputs_sent += re.findall(r"\w+", ''.join(train_ds[i]['dialog'][0]))
  dialog_sent += re.findall(r"\w+", ''.join(train_ds[i]['dialog'][1:]))

  # change to lowercase
  inputs_sent = [x.lower() for x in inputs_sent]
  dialog_sent = [x.lower() for x in dialog_sent]

  inputs_sent.append("<eos>")
  dialog_sent.append("<eos>")

  if len(inputs_sent) >= MAX_SENT_LEN:
    inputs_sent = inputs_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(inputs_sent)):
      inputs_sent.append("<pad>")

  if len(dialog_sent) >= MAX_SENT_LEN:
    dialog_sent = dialog_sent[:MAX_SENT_LEN]
  else:
    for _ in range(MAX_SENT_LEN - len(dialog_sent)):
      dialog_sent.append("<pad>")

  # add parsed sentences
  inputs.append(inputs_sent)
  outputs.append(dialog_sent)

  vocabulary.update(inputs_sent)
  vocabulary.update(dialog_sent)

vocabulary = list(vocabulary)

  0%|          | 0/11118 [00:00<?, ?it/s]

100%|██████████| 11118/11118 [00:02<00:00, 4036.26it/s]


In [6]:
UNK_IDX = vocabulary.index("<unk>") if "<unk>" in vocabulary else 0

In [7]:
# encode each token into index

for i in tqdm(range(len(inputs))):
    inputs[i] = [vocabulary.index(x) if x in vocabulary else UNK_IDX for x in inputs[i]]
    outputs[i] = [vocabulary.index(x) if x in vocabulary else UNK_IDX for x in outputs[i]]

100%|██████████| 11118/11118 [00:45<00:00, 243.31it/s]


In [8]:
# encode each token into index

for i in tqdm(range(len(inputs_test))):
  inputs_test[i] = [vocabulary.index(x) if x in vocabulary else UNK_IDX for x in inputs_test[i]]
  outputs_test[i] = [vocabulary.index(x) if x in vocabulary else UNK_IDX for x in outputs_test[i]]

100%|██████████| 1000/1000 [00:05<00:00, 188.33it/s]


In [14]:
class DialogDS(torch.utils.data.Dataset):
    def __init__(self, inputs, outputs):
        self.x = np.array(inputs, dtype=int)
        self.y = np.array(outputs, dtype=int)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [15]:
train_ds = DialogDS(inputs, outputs)
test_ds = DialogDS(inputs_test, outputs_test)

In [16]:
train_dl = torch.utils.data.DataLoader(train_ds, 32, True)
test_dl = torch.utils.data.DataLoader(test_ds, 32, False)

In [17]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, hidden_size, position_embeddings_size):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size,
                                             hidden_size)
        self.position_embeddings = nn.Embedding(position_embeddings_size,
                                                hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids):
        # Create position IDs for input sequence
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long).repeat(input_ids.size(0), 1).to(DEVICE)
        # Create token and position embeddings
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        # Combine token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [18]:
class TransformerEncoderDecoder(nn.Module):
    def __init__(self, ntoken_src, ntoken_tgt, embedding_dim, position_embeddings_dim, nhead, hidden_size, nlayers, dropout=0.5):
        super(TransformerEncoderDecoder, self).__init__()
        self.model_type = 'Transformer'
        # Embeddings layers
        self.enc_embedding = Embeddings(ntoken_src, embedding_dim, position_embeddings_dim)
        self.dec_embedding = Embeddings(ntoken_tgt, embedding_dim, position_embeddings_dim)
        # Encoder
        encoder_layers = TransformerEncoderLayer(embedding_dim, nhead, hidden_size, dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)
        # Decoder
        decoder_layers = TransformerDecoderLayer(embedding_dim, nhead, hidden_size, dropout)
        self.transformer_decoder = TransformerDecoder(decoder_layers, nlayers)
        # Layer FeedForward
        self.dense = nn.Linear(embedding_dim, ntoken_tgt)
        self.log_softmax = nn.LogSoftmax()

    def forward(self, src, tgt):
      src, tgt = self.enc_embedding(src).permute(1, 0, 2), self.dec_embedding(tgt).permute(1, 0, 2)
      memory = self.transformer_encoder(src)
      transformer_out = self.transformer_decoder(tgt, memory)
      final_out = self.dense(transformer_out)
      return self.log_softmax(final_out)


In [19]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerEncoderDecoder(
    ntoken_src=len(vocabulary),
    ntoken_tgt=len(vocabulary),
    embedding_dim=256,
    position_embeddings_dim=MAX_SENT_LEN,
    nhead=8,
    hidden_size=512,
    nlayers=4,
    dropout=0.1
).to(DEVICE)

loss_fn = nn.NLLLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

c:\Repositories\proai\course9-genai\.venv\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [ ]:
%%time
loss_trace = []
for epoch in tqdm(range(50)):
  current_loss = 0
  for i, (x, y) in enumerate(train_dl):
    x, y  = x.to(DEVICE), y.to(DEVICE)
    outputs = model(x, y)
    loss = loss_fn(outputs.permute(1, 2, 0), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    current_loss += loss.item()
    
  loss_trace.append(current_loss)
  print(f"Epoch {epoch}\tCurrent Loss: {current_loss}")